<a href="https://colab.research.google.com/github/Moe-phantom/boa-constrictor/blob/main/Copy_of_Untitled7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip uninstall -y torch torchaudio torchvision
!pip install --pre torch --index-url https://download.pytorch.org/whl/nightly/cu121 -q
!pip install --pre torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu121 -q
!pip install constriction==0.4.1 pybind11==3.0.1 \
  pyyaml tqdm numpy scipy scikit-learn -q

In [ ]:
!git clone https://github.com/Moe-phantom/boa-constrictor.git
%cd boa-constrictor

In [ ]:
with open('model.py', 'r') as f:
    content = f.read()
content = content.replace(
    'IS_CUDA = torch.cuda.is_available() and device == "cuda"',
    'IS_CUDA = False  # Patched: using GRU backbone instead'
)
with open('model.py', 'w') as f:
    f.write(content)
print("model.py patched")

In [ ]:
# Read current model.py
with open('model.py', 'r') as f:
    content = f.read()

# Replace the IS_CUDA check to force CPU path
# This lets boa.py import without needing mamba_ssm
content = content.replace(
    'IS_CUDA = torch.cuda.is_available() and device == "cuda"',
    'IS_CUDA = False  # Patched: using GRU backbone instead'
)

with open('model.py', 'w') as f:
    f.write(content)

print("model.py patched — mamba_ssm import bypassed")

In [ ]:
model_gru_code = '''
import torch
import torch.nn as nn

def BoaConstrictorGRU(d_model=256, num_layers=4, vocab_size=256, device="cuda"):
    IS_CUDA = torch.cuda.is_available() and device == "cuda"
    device = "cuda" if IS_CUDA else "cpu"

    class GRUBlock(nn.Module):
        def __init__(self, d_model):
            super().__init__()
            self.ln1 = nn.LayerNorm(d_model)
            self.gru = nn.GRU(input_size=d_model, hidden_size=d_model,
                              num_layers=1, batch_first=True)
            self.ln2 = nn.LayerNorm(d_model)
            self.ff = nn.Sequential(
                nn.Linear(d_model, 4 * d_model),
                nn.GELU(),
                nn.Linear(4 * d_model, d_model),
            )
        def forward(self, x, inference_params=None):
            y = self.ln1(x)
            y, _ = self.gru(y)
            y = self.ln2(y)
            y = self.ff(y)
            return x + y
        def init_cache(self, batch_size, device):
            return torch.zeros(1, batch_size, self.gru.hidden_size, device=device)
        def step(self, x, hidden):
            y = self.ln1(x)
            y, hidden = self.gru(y.unsqueeze(1), hidden)
            y = y.squeeze(1)
            y = self.ln2(y)
            y = self.ff(y)
            return x + y, hidden

    class BoaByteGRUPredictor(nn.Module):
        def __init__(self, d_model=256, num_layers=4, vocab_size=256):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, d_model)
            self.blocks = nn.ModuleList([GRUBlock(d_model) for _ in range(num_layers)])
            self.head = nn.Sequential(
                nn.Linear(d_model, d_model),
                nn.ReLU(),
                nn.Linear(d_model, vocab_size)
            )
        def forward(self, x, inference_params=None):
            h = self.embedding(x)
            for blk in self.blocks:
                h = blk(h)
            return self.head(h)
        @torch.inference_mode()
        def init_stream(self, max_len, batch_size=1, device=None, dtype=None):
            d = device or ("cuda" if IS_CUDA else "cpu")
            return [blk.init_cache(batch_size, d) for blk in self.blocks]
        @torch.inference_mode()
        def step(self, byte_t, caches):
            h = self.embedding(byte_t)
            for i, blk in enumerate(self.blocks):
                h, caches[i] = blk.step(h, caches[i])
            return self.head(h)

    return BoaByteGRUPredictor(d_model=d_model, num_layers=num_layers, vocab_size=vocab_size)
'''
with open('model_gru.py', 'w') as f:
    f.write(model_gru_code)
print("model_gru.py written")

In [ ]:
import os
# Remove conflicting cuDNN from library path
os.environ['LD_LIBRARY_PATH'] = ''

import torch
print(torch.cuda.is_available())
print(torch.backends.cudnn.version())

In [ ]:
import os
os.environ['LD_LIBRARY_PATH'] = ''

!pip install torch==2.4.0 \
  --index-url https://download.pytorch.org/whl/cu121 -q --force-reinstall

print("Done — now restart runtime")

In [ ]:
import torch, time, os, sys
sys.path.insert(0, '/content/boa-constrictor')

from model_gru import BoaConstrictorGRU
from boa import BOA

device = "cuda" if torch.cuda.is_available() else "cpu"
data_path = '/content/boa-constrictor/experiments/cms_experiment/CMS_DATA_float32.bin'
output_path = '/content/boa-constrictor/experiments/cms_experiment/cms_gru.boa'

original_size = os.path.getsize(data_path)
print(f"Original: {original_size:,} bytes ({original_size/1024/1024:.2f} MB)")

model = BoaConstrictorGRU(d_model=256, num_layers=4, device=device)
model = model.to(device)
model.eval()
print(f"GRU model ready on {device}")

boa_file = BOA(device=device, filepath=output_path, model=model)

start = time.time()
boa_file.compress(data_path, chunks_count=100)
compress_time = time.time() - start

compressed_size = os.path.getsize(output_path)
ratio = original_size / compressed_size
throughput = (original_size/1024/1024) / compress_time

print(f"\n=== RESULTS ===")
print(f"Original:   {original_size/1024/1024:.2f} MB")
print(f"Compressed: {compressed_size/1024/1024:.2f} MB")
print(f"Ratio:      {ratio:.4f}x")
print(f"Time:       {compress_time:.2f}s")
print(f"Throughput: {throughput:.4f} MB/s")

In [ ]:
model_lstm_code = '''
import torch
import torch.nn as nn

def BoaConstrictorLSTM(d_model=256, num_layers=4, vocab_size=256, device="cuda"):
    IS_CUDA = torch.cuda.is_available() and device == "cuda"
    device = "cuda" if IS_CUDA else "cpu"

    class LSTMBlock(nn.Module):
        def __init__(self, d_model):
            super().__init__()
            self.ln1 = nn.LayerNorm(d_model)
            self.lstm = nn.LSTM(input_size=d_model, hidden_size=d_model,
                               num_layers=1, batch_first=True)
            self.ln2 = nn.LayerNorm(d_model)
            self.ff = nn.Sequential(
                nn.Linear(d_model, 4 * d_model),
                nn.GELU(),
                nn.Linear(4 * d_model, d_model),
            )
        def forward(self, x, inference_params=None):
            y = self.ln1(x)
            y, _ = self.lstm(y)
            y = self.ln2(y)
            y = self.ff(y)
            return x + y
        def init_cache(self, batch_size, device):
            h = torch.zeros(1, batch_size, self.lstm.hidden_size, device=device)
            c = torch.zeros(1, batch_size, self.lstm.hidden_size, device=device)
            return (h, c)
        def step(self, x, hidden):
            y = self.ln1(x)
            y, hidden = self.lstm(y.unsqueeze(1), hidden)
            y = y.squeeze(1)
            y = self.ln2(y)
            y = self.ff(y)
            return x + y, hidden

    class BoaByteGRUPredictor(nn.Module):
        def __init__(self, d_model=256, num_layers=4, vocab_size=256):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, d_model)
            self.blocks = nn.ModuleList([LSTMBlock(d_model) for _ in range(num_layers)])
            self.head = nn.Sequential(
                nn.Linear(d_model, d_model),
                nn.ReLU(),
                nn.Linear(d_model, vocab_size)
            )
        def forward(self, x, inference_params=None):
            h = self.embedding(x)
            for blk in self.blocks:
                h = blk(h)
            return self.head(h)
        @torch.inference_mode()
        def init_stream(self, max_len, batch_size=1, device=None, dtype=None):
            d = device or ("cuda" if IS_CUDA else "cpu")
            return [blk.init_cache(batch_size, d) for blk in self.blocks]
        @torch.inference_mode()
        def step(self, byte_t, caches):
            h = self.embedding(byte_t)
            for i, blk in enumerate(self.blocks):
                h, caches[i] = blk.step(h, caches[i])
            return self.head(h)

    return BoaByteGRUPredictor(d_model=d_model, num_layers=num_layers, vocab_size=vocab_size)
'''
with open('model_lstm.py', 'w') as f:
    f.write(model_lstm_code)
print("model_lstm.py written")

In [ ]:
import os, time, sys
os.environ['LD_LIBRARY_PATH'] = ''
import torch
torch.backends.cudnn.enabled = False
sys.path.insert(0, '/content/boa-constrictor')

from model_lstm import BoaConstrictorLSTM
from boa import BOA

device = "cuda"
data_path = '/content/boa-constrictor/experiments/cms_experiment/CMS_DATA_float32.bin'
output_path = '/content/boa-constrictor/experiments/cms_experiment/cms_lstm.boa'
original_size = os.path.getsize(data_path)

model = BoaConstrictorLSTM(d_model=256, num_layers=4, device=device)
model = model.to(device)
model.eval()
print(f"LSTM model — {sum(p.numel() for p in model.parameters()):,} parameters")

boa_file = BOA(device=device, filepath=output_path, model=model)
start = time.time()
boa_file.compress(data_path, chunks_count=100)
compress_time = time.time() - start

compressed_size = os.path.getsize(output_path)
ratio = original_size / compressed_size
throughput = (original_size/1024/1024) / compress_time

print(f"\n=== LSTM RESULTS ===")
print(f"Original:   {original_size/1024/1024:.2f} MB")
print(f"Compressed: {compressed_size/1024/1024:.2f} MB")
print(f"Ratio:      {ratio:.4f}x")
print(f"Time:       {compress_time:.2f}s")
print(f"Throughput: {throughput:.4f} MB/s")